In [1]:
try:                      # 일단 이걸 시도해본다
    import google.colab   
    # google colab 전용 도구 가져오기 시도 

    IS_COLAB = True       
    # 성공--->colab이다 (스위치 변수)
except ImportError:       
# 시도가 실패하면 아래를 실행, 에러로 멈추지 않게 하는 코드
    IS_COLAB = False      
# 실패---> colab이 아니다 (스위치 변수) 

#colab과 로컬 양쪽에서 돌리고 싶을때 필요


!pip install -q openai python-dotenv  
#!는 컴퓨터 명령어, openai와 dotenv 패키지 설치, -q는 실행 시 표시 되지 않게 조용히 설정

import os   
# 운영체제 기능(환경변수 등)을 다루는 도구

if IS_COLAB: 
    # colab 환경이면 아래 실행
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") 
    # 보관함 키를 환경변수에 올린다. 
    # 환경변수 이름은 대소문자 구분
else: 
# colab 환경이 아니면 아래 실행
    from dotenv import load_dotenv
    load_dotenv()         #env파일을 읽어 환경변수에 올림  

#키 올리기 자체는 필수 / if/else 분기는 환경이 하나뿐이면 필요 X


api_key = os.environ.get("OPENAI_API_KEY","") 
# 키를 읽고 api key 가져옴, **없으면 빈 문자열**로 돌려줌
# 키가 잘 들어왔는지 확인하는 점검용

print(f"api_key: {api_key[:7]}..." if api_key else "키를 못찾았어요")
# f-string: {} 안의 값이 문장에 들어감 / 키가 있으면 앞 7글자만, 없으면 경고 문구 출력


api_key: sk-proj...



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from openai import OpenAI  # 설치한 openai 라이브러리에서 openai 가져오기
client = OpenAI() # OpenAI에 연결하는 클라이언트 생성 후 client라는 이름 부여
                  # 빈칸은 api_key를 자동으로 가져오기 때문에
print("openai 클라이언트 준비완료")

openai 클라이언트 준비완료


In [3]:
response = client.chat.completions.create(   #채팅방식으로 답변을 만들어달라는 요청
    model="gpt-4o-mini",    # 어떤 모델을 사용할지 지정
    messages=[
        {"role": "user", "content": "서울역은 몇호선이야?"}
    ]
) # 메시지, role은 말한 사람, content는 말한 내용
print("response의 타입:", type(response)) # openai가 보낸 응답을 담고 type도 알려줌 
answer = response.choices[0].message.content 
# 답변을 꺼내오는 경로 
# choices는 답변 후보 목록, [0]은 목록 중 첫번째
# message는 답변 메시지, content는 메시지 안의 본문

print("답변:", answer)


response의 타입: <class 'openai.types.chat.chat_completion.ChatCompletion'>
답변: 서울역은 1호선과 4호선이 지나는 역입니다. 1호선은 서울의 주요 지하철 노선 중 하나이며, 4호선은 서울의 동서 방향을 연결하는 노선입니다.


In [4]:
def ask_llm(messages, temperature = 0.3, max_tokens = 500): #ask_llm이라는 이름의 함수 정의
    """메시지 리스트를 받아서 LLM의 답변 텍스트를 돌려준다."""  #함수설명서, 실행에는 영향 없음
    response = client.chat.completions.create(              #채팅방식으로 답변을 만들어달라는 요청
        model="gpt-4o-mini",                  # 어떤 모델을 사용할지 지정       
        messages=messages,                   
        temperature=temperature,
        max_tokens=max_tokens               #messages, temperature, max_tokens는 함수에 필요한 변수
    )
    return response.choices[0].message.content # 답변 텍스트만 반환


In [5]:
question = "서울역은 몇호선이야?" #질문을 변수에 담아 두 호출이 똑같은 질문을 쓰게함

answer_a = ask_llm([{"role": "user", "content": question}])
#메시지가 user 하나, 약 or 강한 인공지능처럼 교과서 같은 답

answer_b = ask_llm([{"role": "system", "content": "당신은 어르신입니다. 상대를 '자네'라 부르며 '~하네, ~라네, ~일세, ~하게나, ~하는가?' 같은 하게체로만 대답"},
                    {"role": "user", "content":question}])
# system 추가, 설정한 형태로 답변

print("기본 답변:\n", answer_a)
print("\n" + "="*50 + "\n") 
# \n은 줄바꿈, =*n--> =를 n개만큼 집어넣어 구분선으로 만듬 

print("어르신 페르소나:\n", answer_b)

기본 답변:
 서울역은 1호선과 4호선이 지나는 역입니다. 1호선은 서울의 주요 기차역 중 하나로, 4호선은 서울 지하철의 노선 중 하나입니다.


어르신 페르소나:
 서울역은 1호선과 4호선이 지나가는 곳이네. 1호선은 남쪽으로 가면 신창역까지 가고, 4호선은 오이도까지 가는 노선이란다네.


In [6]:
messages = [
    {"role": "system", "content": "당신은 친절한 한국어 비서입니다"},
    {"role": "user", "content": "제 이름은 준입니다"},
    {"role": "assistant", "content": "안녕하세요, 준님. 무엇을 도와드릴까요?"},
    {"role": "user", "content": "제 이름이 뭐였죠?"}]
print(ask_llm(messages))

# llm이 답변할 수 있도록 한 대화 녹취록

준님이시죠! 다른 질문이나 도움이 필요하시면 말씀해 주세요.


In [10]:
personas = {
    "초등학교 선생님" : "당신은 초등학생에게 설명하는 선생님입니다. ...",
    "친한 친구" : "...",
    "군인" : "...",
    "어르신" : "...하게체로만 대답"}
question = "서울역에서 가산디지털단지역까지 어떻게 가요?"

# persona는 딕셔너리 {"이름": "지시문"}
# "이름"은 키, "지시문"은 값(내용)

In [11]:
for name, system_prompt in personas.items():
    # 딕셔너리(이름,지시문)을 꺼냄
    # for name, system_prompt --> 딕셔너리에서 꺼내서 아래의 부분을 반복
    answer = ask_llm([ # ask_llm--> 지시문만 바뀌고 질문은 동일
        {"role": "system", "content": system_prompt},
        {"role" : "user", "content": question},])
    print(f"\n[{name}]\n{answer}\n" + "-"*50)


[초등학교 선생님]
서울역에서 가산디지털단지역까지 가는 방법을 알려줄게요!

1. **지하철 이용하기**:
   - 서울역에서 1호선(파란색) 지하철을 타세요.
   - 1호선에서 신도림역으로 가는 방향으로 가면 됩니다.
   - 신도림역에서 2호선(녹색)으로 갈아타세요.
   - 2호선에서 가산디지털단지역 방향으로 가면 됩니다.
   - 가산디지털단지역에 도착하면 하차하면 됩니다!

2. **버스 이용하기**:
   - 서울역 근처에서 가산디지털단지역으로 가는 버스를 찾을 수 있어요.
   - 서울역에서 출발하는 버스 노선 중 가산디지털단지역을 지나가는 버스를 확인하고 탑승하세요.
   - 버스에서 가산디지털단지역에 도착하면 하차하면 됩니다!

지하철이 더 빠르고 편리할 수 있으니, 지하철을 추천해요! 안전하게 잘 가세요!
--------------------------------------------------

[친한 친구]
서울역에서 가산디지털단지역까지 가는 방법은 여러 가지가 있지만, 가장 일반적인 방법은 지하철을 이용하는 것입니다. 아래는 지하철을 이용한 경로입니다.

1. **서울역에서 1호선 탑승**: 서울역에서 1호선(파란색) 지하철을 타고 신용산역 방향으로 가세요.
2. **신용산역에서 7호선으로 환승**: 신용산역에서 하차한 후, 7호선(올리브색)으로 환승합니다.
3. **가산디지털단지역으로 이동**: 7호선을 타고 가산디지털단지역까지 가세요. 가산디지털단지역은 7호선의 역 중 하나입니다.

소요 시간은 약 30분 정도 걸릴 수 있습니다. 지하철 시간표나 혼잡도에 따라 다소 차이가 있을 수 있으니 참고하시기 바랍니다. 안전한 여행 되세요!
--------------------------------------------------

[군인]
서울역에서 가산디지털단지역까지 가는 방법은 다음과 같습니다:

1. **지하철 이용하기**:
   - 서울역에서 1호선(청색)을 타고 신용산역 방향으로 가세요.
   - 신용산역에서 4호선(파란색)으로 

In [12]:
def classify_inquiry_v0(text):
    return ask_llm(
        [
            {"role": "system", "content": "결제/배송/환불/기타"},
            {"role": "user", "content": text},
        ],
        temperature = 1.2,
        max_tokens = 500
    )
for i in range(3):   # 3번 반복
    print(f"{i+1}회차: {classify_inquiry_v0('어제 결제가 두번 됐어요')}")
    # {i+1}-->0부터 시작, 1회차..2회차..3회차로 표시


1회차: 결제가 두 번 된 경우, 먼저 결제 내역을 확인해보시는 것이 좋습니다. 만약 동일한 상품이나 서비스에 대해 중복 결제가 발생했다면, 해당 서비스나 제품을 제공하는 곳에 문의하여 상황을 설명하고 환불 요청을 하시면 됩니다. 

고객 지원센터에 연락하실 때, 결제 관련 정보(날짜, 금액, 결제 방식 등)를 준비해주시면 도움이 될 것입니다. 추가적으로 확인하고 싶은 내용이나 도움이 필요하시면 알려주세요!
2회차: 결제가 두 번 된 경우에는 다음과 같은 조치를 취해보시는 것이 좋습니다:

1. 결제 내역 확인: 사용하신 카드사 또는 결제 서비스의 항목에서 실제로 두 번 결제가 되었는지 확인하세요.
2. 고객센터에 문의: 결제를 진행한 상점의 고객센터에 문의하여 문제를 설명하고 환불 요청을 하세요. 보통 고객센터에서 필요한 정보를 안내해 줄 것입니다.
3. 증빙자료 준비: 결제가 두 번 되었음을 증명할 수 있는 자료(예: 영수증, 결제 내역 화면 캡처 등)를 챙기세요.

문제가 조속히 해결되길 바랍니다!
3회차: 어제 결제가 두 번 되었다면, 아래의 절차를 따라 문제를 해결해 보세요:

1. **구매 내역 확인**: 주문 확인 이메일이나 계정 내 구매 내역을 확인하여 두 번의 결제가 실제로 발생했는지 확인합니다.

2. **고객센터에 연락**: 해당 쇼핑몰이나 서비스의 고객센터에 연락하여 상황을 설명합니다. 주문 번호와 결제 내역을 준비하면 도움이 됩니다.

3. **환불 요청**: 결제가 중복된 경우, 환불 요청을 하세요. 일반적으로 고객센터에서 절차를 안내해 줄 것입니다.

4. **증빙자료 준비**: 필요할 경우, 결제영수증이나 화면 캡처 등의 증빙자료를 준비하면 요청이 빠르게 처리될 수 있습니다.

5. **결과 확인**: 환불 요청이 처리된 후 확인 이메일 또는 안내를 기다리세요.

추가적으로 궁금한 사항이 있다면 언제든지 문의해 주세요!


In [13]:
prompt = "데이터 분석을 한 줄로 설명해주세요"
for temp in [0.0,0.7,1.2]:  # 하나씩 넣어 3번 호출
    answer = ask_llm(
        [{"role":"user", "content":prompt}],
        temperature = temp, #매번 0.0 / 0.7/ 1.2로 바뀜
    )
    print(f"temperature={temp}\n -> {answer}\n")

temperature=0.0
 -> 데이터 분석은 데이터를 수집, 정리, 해석하여 유용한 정보를 도출하고 의사결정을 지원하는 과정입니다.

temperature=0.7
 -> 데이터 분석은 데이터를 수집, 정리, 해석하여 유의미한 정보를 도출하고 의사 결정을 지원하는 과정입니다.

temperature=1.2
 -> 데이터 분석은 데이터를 수집, 정리, 해석하여 의미 있는 정보를 도출하고 의사 결정을 지원하는 과정입니다.



In [14]:
short = ask_llm([{"role": "user", "content": prompt}], max_tokens = 30)
long = ask_llm([{"role": "user", "content":prompt}], max_tokens = 300)

# 같은 질문을 30토큰, 300토큰 한도로 호출

In [15]:
price_input = 0.15    #입력 시 비용
price_output = 0.60   # 출력 시 비용
usd_to_krw = 1400     # 환율

def cost_korean_won(usage):
    input_cost = usage.prompt_tokens / 1_000_000*price_input     # 100만 토큰당 가격 = 입력토큰의 달러비용
    output_cost = usage.completion_tokens / 1_000_000*price_output   # 100만 토큰당 가격 = 출력토큰의 달러비용
    return (input_cost + output_cost) * usd_to_krw    # (입력비+출력비) * 1400 --> 달러를 원화로 계산

In [16]:
# 도전과제

candidates = {
    "A": "회의록을 요약하세요.",
    "B": "당신은 회의록 요약 비서입니다. 3줄로 요약하세요.",
    "C": ("당신은 회의록 요약 비서입니다. 정확히 3줄로 요약하되, "
          "각 줄은 (결정사항) / (담당자) / (마감일) 형식을 따르세요. "
          "회의록에 없는 내용은 절대 추측하지 마세요."),
}

sample_notes = """
[2026-05-14 팀 미팅]
- 김PM: 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기 제안. 이유는 평가 데이터 부족.
- 이개발: 6월 15일이면 가능. 다만 RAG 평가셋 100건 추가 수집 필요.
- 박마케팅: 출시 연기 시 광고 일정도 조정해야 함. 다음 주 월요일까지 결정 필요.
"""

# 세 후보 모두 비교 실행
for key, system_msg in candidates.items():        # 짝을 하나씩 꺼내서 3번 반복
    result = ask_llm([                            # ask_llm = 헬퍼 함수 / 메시지 리스트를 받아 답변 글자 돌려줌
        {"role": "system", "content": system_msg},  # system 메시지, 변수에 담긴 값 사용
        {"role": "user", "content": sample_notes},  # user 메시지
    ], temperature=0)
    print(f"=== 후보 {key} ===")                     # ===후보 A=== 출력
    print(result)                                   # 받은 답변 출력
    print()                                         # 빈줄 출력(가독성을 위해)

=== 후보 A ===
**회의록 요약 (2026-05-14 팀 미팅)**

- **김PM**: 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기 제안, 이유는 평가 데이터 부족.
- **이개발**: 6월 15일 출시 가능, 그러나 RAG 평가셋 100건 추가 수집 필요.
- **박마케팅**: 출시 연기 시 광고 일정 조정 필요, 다음 주 월요일까지 결정 필요.

=== 후보 B ===
김PM은 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기할 것을 제안하며, 평가 데이터 부족을 이유로 들었습니다. 이개발은 6월 15일 출시가 가능하나 RAG 평가셋 100건 추가 수집이 필요하다고 언급했습니다. 박마케팅은 출시 연기에 따라 광고 일정 조정이 필요하며, 다음 주 월요일까지 결정을 내려야 한다고 강조했습니다.

=== 후보 C ===
(신규 챗봇 출시일을 6월 15일로 연기) / (김PM, 이개발, 박마케팅) / (다음 주 월요일)

